#  **Data Collection and Preprocessing**

### Import Libraries

In [39]:
import sys
import os

# Add parent directory to path so we can import from src
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, parent_dir)

In [40]:
from src.preprocessing import (
    clean_review_text,
    display_app_info,
    review_dataframe,
    remove_duplicates,
    handle_missing_data,
    normalize_dates,
    validate_rating,
    preprocessing_report,
    save_cleaned_data,
    count_review_languages,
    remove_non_english_reviews
)
from src.data_scrapping import scrap_reviews

### Web Scraping

#### App metadata

In [41]:
DAHSEN_APP_ID = 'com.dashen.dashensuperapp'
display_app_info(DAHSEN_APP_ID)

Dashen Bank App Info
App Title   : Dashen Bank
Current Score: 4.2310405
Total Ratings: 5,630
Total Reviews: 1,023
Installs     : 1,000,000+


#### Scrape reviews

In [42]:
reviews = scrap_reviews(app_id=DAHSEN_APP_ID, num_reviews=1000)

Scraping reviews for com.dashen.dashensuperapp...
Collected 1000 raw reviews


#### Collect review text, rating, review date, bank , source

In [43]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(reviews[0].keys()))

print("\nFirst raw review (sample):")
for key, value in reviews[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 59a20f9e-bb87-4d5f-bee7-ce19f19baddf
  userName: Bellixs
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjWdkI8XakUGuA_kKU_HQlx-U8YwqF17e6QyzaymZxmEE88LRWLk
  content: good
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 1.8.1
  at: 2026-05-14 17:35:20
  replyContent: None
  repliedAt: None
  appVersion: 1.8.1


In [44]:
df = review_dataframe(reviews, app_info={'title': 'Dashen Bank'})

print(f"Shape: {df.shape}")
df.head()

Shape: (1000, 6)


,review_id,review,rating,date,bank,source
0,59a20f9e-bb87-4d5f-bee7-ce19f19baddf,good,5,2026-05-14 17:35:20,Dashen Bank,Google Play
1,fc184115-ab13-482b-bb08-798589a4d482,good app but it was doesnt work other bank tra...,5,2026-05-14 17:29:26,Dashen Bank,Google Play
2,105aa71e-1e72-4618-899b-78a321a5258b,good,5,2026-05-14 16:15:23,Dashen Bank,Google Play
3,3cf7873a-56b2-4e5d-b963-eb31a72d3fc7,"i swear to god , By using this app, I won a Sa...",5,2026-05-14 15:58:59,Dashen Bank,Google Play
4,fd76bc39-9ee1-4a56-b784-3c6e46f83d39,good and easier to used,5,2026-05-14 15:19:29,Dashen Bank,Google Play


## Preprocessing

#### Remove duplicate reviews

In [45]:
df_clean = df.copy()

In [46]:
df_clean = remove_duplicates(df_clean)

Removed 0 duplicate reviews
Remaining: 1000 reviews


#### Handle missing values

In [47]:
df_clean =handle_missing_data(df_clean)

Removed 0 rows with missing critical data
Remaining: 1000 reviews


#### Normalize dates to YYYY-MM-DD format

In [48]:
df_clean = normalize_dates(df_clean)

Before normalization:
0   2026-05-14 17:35:20
1   2026-05-14 17:29:26
2   2026-05-14 16:15:23
dtype: datetime64[us]

After normalization:
0    2026-05-14
1    2026-05-14
2    2026-05-14
dtype: str

Date range: 2025-01-14 to 2026-05-14


#### Handle incorrect ratings

In [49]:
df_clean = validate_rating(df_clean)

All ratings are valid (1-5).
Remaining: 1000 reviews


#### Clean review text

In [50]:
df['review'] = df['review'].apply(clean_review_text)

print("Sample cleaned reviews:")
print(df['review'].head(10).to_string())

Sample cleaned reviews:
0                                                 good
1    good app but it was doesnt work other bank tra...
2                                                 good
3    i swear to god , by using this app, i won a sa...
4                              good and easier to used
5                                                  1  
6                            bad mobile banking at all
7                                       very nice app.
8                                   very difficult app
9    good app, but debit transactions not allowed w...


#### Save the cleaned dataset

In [51]:
# Select only the 5 required columns in the right order
df_clean = df_clean[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (1000, 5)


,review,rating,date,bank,source
0,good,5,2026-05-14,Dashen Bank,Google Play
1,good,5,2026-05-14,Dashen Bank,Google Play
2,"i swear to god , By using this app, I won a Sa...",5,2026-05-14,Dashen Bank,Google Play
3,good and easier to used,5,2026-05-14,Dashen Bank,Google Play
4,good app but it was doesnt work other bank tra...,5,2026-05-14,Dashen Bank,Google Play
5,very difficult app,1,2026-05-13,Dashen Bank,Google Play
6,nice,5,2026-05-13,Dashen Bank,Google Play
7,"Good app, but debit transactions not allowed W...",5,2026-05-13,Dashen Bank,Google Play
8,very nice app.,5,2026-05-13,Dashen Bank,Google Play
9,bad mobile banking at all,1,2026-05-13,Dashen Bank,Google Play


In [52]:
save_cleaned_data(df_clean, output_path="../data/processed/boa_reviews_cleaned.csv")

Cleaned data saved to ../data/processed/boa_reviews_cleaned.csv
Saved to: ../data/processed/boa_reviews_cleaned.csv


### Report

In [53]:
preprocessing_report(df, df_clean)

  PREPROCESSING REPORT — Awash Bank Reviews

  Raw reviews collected  :   1000
  Reviews after cleaning :   1000
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-01-14  to  2026-05-14
Rating distribution:
  5 stars:  711  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  4 stars:   61  ████████████
  3 stars:   43  ████████
  2 stars:   39  ███████
  1 stars:  146  █████████████████████████████

  Text length stats:
    Min    : 1 characters
    Median : 27 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

